In [ ]:
# imports
import pandas as pd
import requests
import zipfile
import io
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader, TensorDataset

For this example we will be using the Concrete Compressive Strength Dataset: https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength

In [ ]:
# URL to the concrete+compressive+strength.zip file on the UCI archive
concrete_zip_url = "https://archive.ics.uci.edu/static/public/165/concrete+compressive+strength.zip"

# Download the zip file
response = requests.get(concrete_zip_url)
response.raise_for_status() # Raise an exception for bad status codes

# Read the zip file from memory
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    # Print all file names in the zip archive to identify the correct one

    # Use the correct Excel file name identified from z.namelist()
    with z.open('Concrete_Data.xls') as excel_file:
        concrete_df = pd.read_excel(excel_file)

# create X and y, scale data, create train and test
target_column_name = concrete_df.columns[-1]
y = concrete_df[target_column_name]
X = concrete_df.drop(target_column_name, axis=1)

scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X).astype(np.float32))
y = pd.DataFrame(y.astype(np.float32))

# train / val split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=499)
# NOTE: I haven't used a test set in this example because it's all about learning!

display(X_train.head())
display(y_train.head())

,0,1,2,3,4,5,6,7
316,-0.281032,-0.856886,0.715275,-1.659688,1.029528,0.425670,1.574577,-0.279733
554,-0.411326,0.984546,-0.847132,0.193657,-1.038944,0.870881,-0.490150,-0.612331
663,-1.418445,1.462298,-0.847132,0.488805,-1.038944,-0.585704,0.818867,-0.279733
881,-1.226977,0.824522,0.919448,-0.167080,0.300957,0.374201,-1.055435,-0.279733
774,0.965325,-0.856886,-0.847132,0.207711,-1.038944,1.776742,0.130042,-0.612331


,"Concrete compressive strength(MPa, megapascals)"
316,33.942902
554,15.691094
663,27.874825
881,25.558876
774,11.465986


In [ ]:
class FCNN(nn.Module):
  def __init__(self, input_size):
    super(FCNN, self).__init__()
    self.fc1 = nn.Linear(input_size, 32)
    self.fc2 = nn.Linear(32, 16)
    self.fc3 = nn.Linear(16, 1)
    self.relu = nn.ReLU()

  def forward(self, x):
    x = self.fc1(x)
    x = self.relu(x)
    x = self.fc2(x)
    x = self.relu(x)
    return self.fc3(x)

In [ ]:
# Initialize the models we defined above
model = FCNN(input_size=X_train.shape[1])

# To use mini-batch GD we need to create a Dataset object
X_train_torch = torch.tensor(X_train.values, dtype=torch.float32)
y_train_torch = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
X_val_torch = torch.tensor(X_val.values, dtype=torch.float32)
y_val_torch = torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1)
# Wrap them in a Dataset
train_dataset = TensorDataset(X_train_torch, y_train_torch)
val_dataset = TensorDataset(X_val_torch, y_val_torch)
# Now create a DataLoader for Mini-Batch GD
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Define our Loss function and algorithm for optimization (aka our 'Optimizer')
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=1e-3)

# training loop
epochs = 5_000

for epoch in range(epochs):

    # --- TRAINING PHASE ---
    model.train() # Set to training mode
    train_loss = 0
    for data, target in train_loader:
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        if (epoch + 1) % 50 == 0:
            print(f"Epoch {epoch + 1}/{epochs} | Batch Loss: {loss.item():.4f}")


    # --- VALIDATION PHASE ---
    model.eval() # Set to evaluation mode (turns off Dropout, BatchNorm, etc.)
    val_loss = 0
    with torch.no_grad(): # Disables gradient calculation
        for data, target in val_loader:
            output = model(data)
            loss = criterion(output, target)
            val_loss += loss.item()

    if (epoch + 1) % 50 == 0:
        with torch.no_grad():
            # Calculate average losses
            avg_train = train_loss / len(train_loader)
            avg_val = val_loss / len(val_loader)

            print()
            print(f"Epoch {epoch + 1}: Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")
            print("-----------------------")
            print()

Epoch 50/5000 | Batch Loss: 36.3116
Epoch 50/5000 | Batch Loss: 26.9061
Epoch 50/5000 | Batch Loss: 42.4896
Epoch 50/5000 | Batch Loss: 29.3388
Epoch 50/5000 | Batch Loss: 41.9316
Epoch 50/5000 | Batch Loss: 38.1701
Epoch 50/5000 | Batch Loss: 36.0753
Epoch 50/5000 | Batch Loss: 46.9081
Epoch 50/5000 | Batch Loss: 26.7620
Epoch 50/5000 | Batch Loss: 15.4921
Epoch 50/5000 | Batch Loss: 12.6500
Epoch 50/5000 | Batch Loss: 22.3654
Epoch 50/5000 | Batch Loss: 20.5328
Epoch 50/5000 | Batch Loss: 26.0820
Epoch 50/5000 | Batch Loss: 15.1143
Epoch 50/5000 | Batch Loss: 22.0487
Epoch 50/5000 | Batch Loss: 25.0059
Epoch 50/5000 | Batch Loss: 21.5915
Epoch 50/5000 | Batch Loss: 25.4502
Epoch 50/5000 | Batch Loss: 14.8567
Epoch 50/5000 | Batch Loss: 21.0647
Epoch 50/5000 | Batch Loss: 19.5796
Epoch 50/5000 | Batch Loss: 11.8347
Epoch 50/5000 | Batch Loss: 20.1968
Epoch 50/5000 | Batch Loss: 27.7583
Epoch 50/5000 | Batch Loss: 30.6350

Epoch 50: Train Loss: 26.0443 | Val Loss: 36.1170
-------------

In [ ]:
# using Adam AND mini-batch GD
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# training loop
epochs = 5_000

for epoch in range(epochs):

    # --- TRAINING PHASE ---
    model.train() # Set to training mode
    train_loss = 0
    for data, target in train_loader:
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        if (epoch + 1) % 50 == 0:
            print(f"Epoch {epoch + 1}/{epochs} | Batch Loss: {loss.item():.4f}")

    # --- VALIDATION PHASE ---
    model.eval() # Set to evaluation mode (turns off Dropout, BatchNorm, etc.)
    val_loss = 0
    with torch.no_grad(): # Disables gradient calculation
        for data, target in val_loader:
            output = model(data)
            loss = criterion(output, target)
            val_loss += loss.item()

    if (epoch + 1) % 50 == 0:
        with torch.no_grad():
            # Calculate average losses
            avg_train = train_loss / len(train_loader)
            avg_val = val_loss / len(val_loader)

            print()
            print(f"Epoch {epoch + 1}: Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")
            print("-----------------------")
            print()

Epoch 50/5000 | Batch Loss: 2.9975
Epoch 50/5000 | Batch Loss: 6.5345
Epoch 50/5000 | Batch Loss: 2.1532
Epoch 50/5000 | Batch Loss: 17.5627
Epoch 50/5000 | Batch Loss: 5.3105
Epoch 50/5000 | Batch Loss: 3.5895
Epoch 50/5000 | Batch Loss: 1.4182
Epoch 50/5000 | Batch Loss: 0.9115
Epoch 50/5000 | Batch Loss: 1.0699
Epoch 50/5000 | Batch Loss: 3.0232
Epoch 50/5000 | Batch Loss: 1.7873
Epoch 50/5000 | Batch Loss: 1.0652
Epoch 50/5000 | Batch Loss: 2.2678
Epoch 50/5000 | Batch Loss: 2.0310
Epoch 50/5000 | Batch Loss: 1.6513
Epoch 50/5000 | Batch Loss: 1.3421
Epoch 50/5000 | Batch Loss: 2.1279
Epoch 50/5000 | Batch Loss: 0.9257
Epoch 50/5000 | Batch Loss: 1.4748
Epoch 50/5000 | Batch Loss: 0.7168
Epoch 50/5000 | Batch Loss: 0.8837
Epoch 50/5000 | Batch Loss: 1.8747
Epoch 50/5000 | Batch Loss: 2.4796
Epoch 50/5000 | Batch Loss: 3.0976
Epoch 50/5000 | Batch Loss: 1.8968
Epoch 50/5000 | Batch Loss: 1.8581

Epoch 50: Train Loss: 2.7712 | Val Loss: 21.5242
-----------------------

Epoch 100/5000

In [ ]:
# using a learning rate decay with Adam and mini-batch GD
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# LR Decay
# This will multiply the LR by 0.1 every 20 epochs
scheduler = StepLR(optimizer, step_size=20, gamma=0.1)

# training loop
epochs = 5_000

for epoch in range(epochs):

    # --- TRAINING PHASE ---
    model.train() # Set to training mode
    train_loss = 0
    for data, target in train_loader:
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        if (epoch + 1) % 50 == 0:
            print(f"Epoch {epoch + 1}/{epochs} | Batch Loss: {loss.item():.4f}")

    # --- VALIDATION PHASE ---
    model.eval() # Set to evaluation mode (turns off Dropout, BatchNorm, etc.)
    val_loss = 0
    with torch.no_grad(): # Disables gradient calculation
        for data, target in val_loader:
            output = model(data)
            loss = criterion(output, target)
            val_loss += loss.item()

    if (epoch + 1) % 50 == 0:
        with torch.no_grad():
            # Calculate average losses
            avg_train = train_loss / len(train_loader)
            avg_val = val_loss / len(val_loader)

            print()
            print(f"Epoch {epoch + 1}: Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")
            print("-----------------------")
            print()

Epoch 50/5000 | Batch Loss: 0.4021
Epoch 50/5000 | Batch Loss: 3.0678
Epoch 50/5000 | Batch Loss: 0.2946
Epoch 50/5000 | Batch Loss: 2.3841
Epoch 50/5000 | Batch Loss: 5.0493
Epoch 50/5000 | Batch Loss: 0.3927
Epoch 50/5000 | Batch Loss: 0.9765
Epoch 50/5000 | Batch Loss: 0.9415
Epoch 50/5000 | Batch Loss: 4.3944
Epoch 50/5000 | Batch Loss: 0.5224
Epoch 50/5000 | Batch Loss: 16.7516
Epoch 50/5000 | Batch Loss: 0.6770
Epoch 50/5000 | Batch Loss: 0.4731
Epoch 50/5000 | Batch Loss: 0.2874
Epoch 50/5000 | Batch Loss: 0.2518
Epoch 50/5000 | Batch Loss: 0.3656
Epoch 50/5000 | Batch Loss: 2.0428
Epoch 50/5000 | Batch Loss: 0.4996
Epoch 50/5000 | Batch Loss: 0.7365
Epoch 50/5000 | Batch Loss: 0.6222
Epoch 50/5000 | Batch Loss: 0.3551
Epoch 50/5000 | Batch Loss: 1.4441
Epoch 50/5000 | Batch Loss: 0.6463
Epoch 50/5000 | Batch Loss: 0.4527
Epoch 50/5000 | Batch Loss: 0.8164
Epoch 50/5000 | Batch Loss: 2.6649

Epoch 50: Train Loss: 1.8274 | Val Loss: 27.4898
-----------------------

Epoch 100/5000

Can you use the above information to create a plot to visualize the learning rate improving? You will need to add a 'standard' GD option also.